# Aura Vision — Virtual Try-On Assessment Notebook
This notebook walks through all four tasks from the AI/ML internship assessment with visual outputs.

**Tasks:**
1. Garment Try-On (overlay garment on person)
2. Body Measurement Estimation
3. Product Recommendation Engine
4. Garment Classifier (bonus)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from engine import VastraEngine
from recommender import AuraRecommender

# Initialize our core AI engines
engine = VastraEngine()
recommender = AuraRecommender()

def show_image(image_bytes, title=''):
    """Helper function to display image bytes in the notebook."""
    import numpy as np
    arr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()
    
print("Engines loaded successfully!")

## Tasks 1, 2 & 4 — Virtual Try-On, Measurements, and Classification
**Approach:** 
We use MediaPipe to extract 33 body landmarks and calculate the Euclidean distance between the shoulders to determine real-world measurements based on user height. We then use `rembg` to isolate the garment and apply a 2D affine transformation to overlay it. During this process, we also run the garment through OpenAI's CLIP model (Zero-Shot) to classify its category.

In [ ]:
# NOTE: To run this cell, make sure you have two images named 'person.jpg' and 'garment.jpg' in your folder!
try:
    with open('person.jpg', 'rb') as p, open('garment.jpg', 'rb') as g:
        person_bytes = p.read()
        garment_bytes = g.read()

    # Execute Try-On pipeline (170cm reference height)
    result_image_bytes, info = engine.execute_tryon(person_bytes, garment_bytes, height_cm=170)

    if result_image_bytes:
        show_image(result_image_bytes, "Task 1: AI Fitting Result")
        
        print("\n--- Task 2: Estimated Measurements ---")
        for key, val in info.get('measurements', {}).items():
            print(f"{key.replace('_', ' ').title()}: {val}")
            
        print("\n--- Task 4: Garment Classification ---")
        gc = info.get('garment_class', {})
        print(f"Detected Class: {gc.get('class')}")
        print(f"Confidence: {gc.get('confidence')}%")

except FileNotFoundError:
    print("Please add 'person.jpg' and 'garment.jpg' to your project folder to see the visual output.")

## Task 3 — Product Recommendation Engine
**Approach:** 
The engine filters by community and price range, then utilizes content-based scoring based on category matching and tag-set overlap with past purchases. It combines this relevance score with the product's global rating to generate the final top recommendations.

In [ ]:
# Create a sample user profile
user_profile = {
    "community": "Muslim",
    "preferred_categories": ["Full Body", "Top"],
    "price_range": (1000, 5000),
    "past_purchases": ["eid", "abaya", "embroidery"]
}

# Fetch top 3 personalized recommendations
recs = recommender.get_recommendations(user_profile, top_n=3)

print("--- Task 3: Personalized Recommendations ---\n")
for i, item in enumerate(recs, 1):
    print(f"{i}. {item['name']} (₹{item['price']}) - Rating: {item['rating']}★")
    print(f"   Why: {item['explanation']}\n")